In [1]:
!pip install pytrec_eval

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for pytrec_eval: filename=pytrec_eval-0.5-cp311-cp311-win_amd64.whl size=57111 sha256=69734f53a58edc8b758af11f71c528d0d1a33062282650e75a8f727f144b2896
  Stored in directory: c:\users\acer\appdata\local\packages\pythonsoftwarefoundation.python.3.11_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\0f\89\42\86aecdb99975f1840c27bc37fdfed72116abcf82e2c9dc76a8
Successfully built pytrec_eval


  DEPRECATION: Building 'pytrec_eval' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pytrec_eval'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\ACER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import json
import time
import itertools
import numpy as np
import pytrec_eval
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

In [3]:
CORPUS_PATH  = r"Data\clean_corpus.jsonl"
QUERIES_PATH = r"Data\clean_queries.jsonl"
QRELS_PATH   = r"Data\qrels\train.tsv"
TOP_K = 10  # NDCG@10

In [4]:
PARAM_GRID = {
    "sublinear_tf": [False, True],
    "min_df":       [1, 2, 3],
    "max_df":       [0.85, 0.90, 0.95, 1.0],
    "ngram_range":  [(1, 1), (1, 2)],
    "norm":         ["l2"],           # cosine = dot sau l2-norm → giữ l2
    "smooth_idf":   [True, False],
}

In [5]:
# ==================== LOAD DỮ LIỆU ====================
def load_jsonl(path: str) -> dict:
    data = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                data[obj["_id"]] = obj["text"]
    return data
 
def load_qrels(path: str) -> dict:
    qrels = defaultdict(dict)
    with open(path, encoding="utf-8") as f:
        next(f, None)                          # bỏ header
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) < 3:
                continue
            q_id, c_id, score = parts[0], parts[1], int(parts[2])
            qrels[q_id][c_id] = score
    return dict(qrels)


In [6]:
# ==================== RETRIEVAL + EVAL ====================
def run_retrieval_and_eval(
    vectorizer: TfidfVectorizer,
    doc_matrix,           # sparse (n_docs, vocab) — đã l2-norm
    doc_ids: list,
    queries: dict,
    qrels_dict: dict,
    k: int = 10,
) -> float:
    """
    Trả về NDCG@k trung bình.
    Cosine similarity = dot product vì cả query lẫn doc đều đã l2-norm.
    """
    eval_queries = {q: t for q, t in queries.items() if q in qrels_dict}
    run_dict = {}
 
    for q_id, q_text in eval_queries.items():
        # Vector hoá query rồi l2-norm → cosine = dot product
        q_vec = vectorizer.transform([q_text])
        q_vec = normalize(q_vec, norm="l2")
 
        scores = (doc_matrix @ q_vec.T).toarray().flatten()  # cosine scores
 
        top_idx = np.argsort(scores)[::-1][:k]
        run_dict[q_id] = {doc_ids[i]: float(scores[i]) for i in top_idx}
 
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_dict, {f"ndcg_cut_{k}"}
    )
    results = evaluator.evaluate(run_dict)
    ndcg_scores = [v[f"ndcg_cut_{k}"] for v in results.values()]
    return float(np.mean(ndcg_scores))
 

In [7]:
# ==================== GRID SEARCH ====================
def grid_search(corpus, queries, qrels_dict):
    doc_ids = list(corpus.keys())
    texts   = list(corpus.values())
 
    # Tạo danh sách tất cả tổ hợp
    keys   = list(PARAM_GRID.keys())
    combos = list(itertools.product(*[PARAM_GRID[k] for k in keys]))
    total  = len(combos)
    print(f"Tổng số tổ hợp cần thử: {total}\n")
 
    results_log = []
    best_ndcg   = -1.0
    best_params = None
 
    for idx, combo in enumerate(combos, 1):
        params = dict(zip(keys, combo))
 
        # Bỏ qua tổ hợp vô lý
        if params["min_df"] >= len(texts) * params["max_df"]:
            continue
 
        t0 = time.time()
        try:
            vectorizer = TfidfVectorizer(
                analyzer      = "word",
                token_pattern = r"(?u)\S+",
                use_idf       = True,
                **params
            )
            doc_matrix = vectorizer.fit_transform(texts)
            doc_matrix = normalize(doc_matrix, norm="l2")  # l2-norm sẵn cho cosine
 
            ndcg = run_retrieval_and_eval(
                vectorizer, doc_matrix, doc_ids, queries, qrels_dict, k=TOP_K
            )
        except Exception as e:
            print(f"[{idx:>4}/{total}] LỖI — {params} → {e}")
            continue
 
        elapsed = time.time() - t0
        vocab_size = len(vectorizer.vocabulary_)
 
        log_entry = {**params, "ndcg@10": ndcg, "vocab_size": vocab_size, "time_s": round(elapsed, 2)}
        results_log.append(log_entry)
 
        marker = " ← BEST" if ndcg > best_ndcg else ""
        if ndcg > best_ndcg:
            best_ndcg   = ndcg
            best_params = params.copy()
 
        print(
            f"[{idx:>4}/{total}] NDCG@10={ndcg:.4f}{marker} | "
            f"vocab={vocab_size:,} | {elapsed:.1f}s | {params}"
        )
 
    return best_params, best_ndcg, results_log

In [8]:
def main():
    print("=" * 70)
    print("TF-IDF HYPERPARAMETER SEARCH — NDCG@10 (Cosine Similarity)")
    print("=" * 70)
 
    print("\n[1/3] Load dữ liệu...")
    corpus     = load_jsonl(CORPUS_PATH)
    queries    = load_jsonl(QUERIES_PATH)
    qrels_dict = load_qrels(QRELS_PATH)
    print(f"  Corpus : {len(corpus):,} docs")
    print(f"  Queries: {len(queries):,}")
    print(f"  Qrels  : {len(qrels_dict):,} queries\n")
 
    print("[2/3] Bắt đầu Grid Search...\n")
    t_start = time.time()
    best_params, best_ndcg, results_log = grid_search(corpus, queries, qrels_dict)
    t_total = time.time() - t_start
 
    print("\n[3/3] KẾT QUẢ")
    print("=" * 70)
    print(f"Thời gian tổng : {t_total/60:.1f} phút")
    print(f"Best NDCG@10   : {best_ndcg:.4f}")
    print(f"Best params    : {best_params}")
    print("=" * 70)
 
    # Top 5 tổ hợp
    results_log.sort(key=lambda x: x["ndcg@10"], reverse=True)
    print("\nTop 5 bộ siêu tham số:")
    print(f"{'Rank':<5} {'NDCG@10':<10} {'sublinear':<11} {'min_df':<8} "
          f"{'max_df':<8} {'ngram':<10} {'smooth_idf':<12} {'vocab':>8}")
    print("-" * 75)
    for rank, entry in enumerate(results_log[:5], 1):
        print(
            f"{rank:<5} {entry['ndcg@10']:<10.4f} "
            f"{str(entry['sublinear_tf']):<11} {entry['min_df']:<8} "
            f"{entry['max_df']:<8} {str(entry['ngram_range']):<10} "
            f"{str(entry['smooth_idf']):<12} {entry['vocab_size']:>8,}"
        )
 
    return best_params, best_ndcg, results_log

In [9]:
main()

TF-IDF HYPERPARAMETER SEARCH — NDCG@10 (Cosine Similarity)

[1/3] Load dữ liệu...
  Corpus : 3,633 docs
  Queries: 3,237
  Qrels  : 2,590 queries

[2/3] Bắt đầu Grid Search...

Tổng số tổ hợp cần thử: 96

[   1/96] NDCG@10=0.2260 ← BEST | vocab=52,188 | 4.6s | {'sublinear_tf': False, 'min_df': 1, 'max_df': 0.85, 'ngram_range': (1, 1), 'norm': 'l2', 'smooth_idf': True}
[   2/96] NDCG@10=0.2263 ← BEST | vocab=52,188 | 4.7s | {'sublinear_tf': False, 'min_df': 1, 'max_df': 0.85, 'ngram_range': (1, 1), 'norm': 'l2', 'smooth_idf': False}
[   3/96] NDCG@10=0.2271 ← BEST | vocab=331,615 | 11.0s | {'sublinear_tf': False, 'min_df': 1, 'max_df': 0.85, 'ngram_range': (1, 2), 'norm': 'l2', 'smooth_idf': True}
[   4/96] NDCG@10=0.2277 ← BEST | vocab=331,615 | 12.6s | {'sublinear_tf': False, 'min_df': 1, 'max_df': 0.85, 'ngram_range': (1, 2), 'norm': 'l2', 'smooth_idf': False}
[   5/96] NDCG@10=0.2260 | vocab=52,188 | 4.4s | {'sublinear_tf': False, 'min_df': 1, 'max_df': 0.9, 'ngram_range': (1, 1), '

({'sublinear_tf': True,
  'min_df': 1,
  'max_df': 0.85,
  'ngram_range': (1, 2),
  'norm': 'l2',
  'smooth_idf': False},
 0.23570680710893202,
 [{'sublinear_tf': True,
   'min_df': 1,
   'max_df': 0.85,
   'ngram_range': (1, 2),
   'norm': 'l2',
   'smooth_idf': False,
   'ndcg@10': 0.23570680710893202,
   'vocab_size': 331615,
   'time_s': 11.46},
  {'sublinear_tf': True,
   'min_df': 1,
   'max_df': 0.9,
   'ngram_range': (1, 2),
   'norm': 'l2',
   'smooth_idf': False,
   'ndcg@10': 0.23570680710893202,
   'vocab_size': 331615,
   'time_s': 13.05},
  {'sublinear_tf': True,
   'min_df': 1,
   'max_df': 0.95,
   'ngram_range': (1, 2),
   'norm': 'l2',
   'smooth_idf': False,
   'ndcg@10': 0.23570680710893202,
   'vocab_size': 331615,
   'time_s': 13.59},
  {'sublinear_tf': True,
   'min_df': 1,
   'max_df': 1.0,
   'ngram_range': (1, 2),
   'norm': 'l2',
   'smooth_idf': False,
   'ndcg@10': 0.23570680710893202,
   'vocab_size': 331615,
   'time_s': 16.28},
  {'sublinear_tf': True,
 